In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Load Data
data_dir = Path("../data/raw")
csv_files = list(data_dir.glob("*_daily.csv"))
df = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in csv_files], ignore_index=True)

# 2. Fill Missing Values
# If snowfall is NaN for a hot city, we assume it's 0.
df["snowfall_sum"] = df["snowfall_sum"].fillna(0.0)

# 3. Calendar Features (Global Seasonality)
df["day_of_year"] = df["date"].dt.day_of_year
# Sine/Cosine maps the 365 days into a continuous circle (Dec 31 is close to Jan 1)
df["sin_day"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
df["cos_day"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

# 4. Lag Features (Must be sorted by city and date first!)
df = df.sort_values(by=["city", "date"]).reset_index(drop=True)

# Create lags per city so we don't mix up London's weather with Tokyo's
df["temp_lag_1"] = df.groupby("city")["temperature_2m_mean"].shift(1)
df["temp_lag_2"] = df.groupby("city")["temperature_2m_mean"].shift(2)
df["temp_lag_3"] = df.groupby("city")["temperature_2m_mean"].shift(3)

# 7-day rolling average (useful to capture weekly weather trends)
df["temp_rolling_7"] = df.groupby("city")["temperature_2m_mean"].transform(lambda x: x.rolling(7).mean())

# Drop the first 7 days of each city because they will have NaNs due to the lags
df_clean = df.dropna().reset_index(drop=True)

print(f"Original rows: {len(df)}")
print(f"Clean rows (after lags): {len(df_clean)}")

# 5. Save Processed Data
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(processed_dir / "global_weather_features.csv", index=False)
print(f"Saved to data/processed/global_weather_features.csv")

Original rows: 14608
Clean rows (after lags): 14560
Saved to data/processed/global_weather_features.csv


In [8]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Load processed data
df_clean = pd.read_csv("../data/processed/global_weather_features.csv", parse_dates=["date"])

# Define features (X) and target (y)
features = [
    "latitude", "longitude", "sin_day", "cos_day", 
    "temp_lag_1", "temp_lag_2", "temp_lag_3", "temp_rolling_7"
]
target = "temperature_2m_mean"

X = df_clean[features]
y = df_clean[target]

# Time-based split: Train on older data, test on the most recent 20%
split_idx = int(len(df_clean) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training rows: {len(X_train):,} | Testing rows: {len(X_test):,}")

# Train XGBoost
print("Training XGBoost model...")
model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"\nModel Performance (Global):")
print(f"MAE: {mae:.2f} °C")
print(f"RMSE: {rmse:.2f} °C")

Training rows: 11,648 | Testing rows: 2,912
Training XGBoost model...

Model Performance (Global):
MAE: 1.33 °C
RMSE: 1.77 °C


In [9]:
# Naive baseline: predict today's temp (lag 1) for tomorrow
persistence_predictions = X_test["temp_lag_1"]
baseline_mae = mean_absolute_error(y_test, persistence_predictions)

print(f"Persistence Baseline MAE: {baseline_mae:.2f} °C")
print(f"XGBoost Model MAE:        {mae:.2f} °C")

if mae < baseline_mae:
    print(f"\n✅ Success! XGBoost is {baseline_mae - mae:.2f}°C better than the baseline.")
else:
    print(f"\n⚠️ The model didn't beat the baseline.")

Persistence Baseline MAE: 1.45 °C
XGBoost Model MAE:        1.33 °C

✅ Success! XGBoost is 0.13°C better than the baseline.


In [10]:
import joblib
from pathlib import Path

# Create models directory if it doesn't exist
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save the model
model_path = models_dir / "global_xgboost_weather.json"
model.save_model(model_path)

print(f"✅ Model saved successfully to {model_path}")

✅ Model saved successfully to ..\models\global_xgboost_weather.json
